# 5-2 Quantization 실습

---

## 학습 개요

### 학습 주제
1. **FP32/FP16의 한계와 Quantization 필요성**: 고정밀도 부동소수점의 메모리/연산 비용을 이해하고 양자화로 경량화하는 방법
2. **PTQ(Post-Training Quantization) 실습**: bitsandbytes 라이브러리를 활용한 INT4 양자화 적용 및 FP16과의 성능 비교
3. **Test-time Prompting(TTP) 전략**: 양자화 후 환경 변화 입력(오타/노이즈/모호함)에서 품질 저하를 프롬프트 강화로 보완하는 기법

### 학습 목표
- FP16과 INT4 모델의 차이(속도/메모리/정확도)를 측정하고 비교할 수 있다
- bitsandbytes를 사용하여 INT4 양자화를 적용할 수 있다
- 환경 변화(오타/노이즈/모호함) 시 모델 품질 저하를 관찰하고 원인을 설명할 수 있다
- TTP 전략을 적용하여 품질 저하를 완화할 수 있다
- 양자화와 TTP의 트레이드오프를 이해하고 실무 적용 시나리오를 설명할 수 있다

### 핵심 개념
1. **Quantization(양자화)**: 높은 정밀도(FP32/FP16)의 가중치를 낮은 정밀도(INT8/INT4)로 변환하여 메모리와 연산량을 줄이는 기법. 수식: `Q(x) = round(x / scale) + zero_point`
2. **PTQ(Post-Training Quantization)**: 학습이 완료된 후 양자화를 적용하는 방식. 재학습 없이 빠르게 적용 가능하나 정확도 손실 가능.
3. **INT4 (NF4)**: 4비트 표현으로 FP16(16비트) 대비 메모리 약 75% 감소. bitsandbytes의 NF4는 LLM 가중치 분포에 최적화된 양자화 방식.
4. **Test-time Prompting(TTP)**: 추론 시점에 프롬프트를 강화하여 모델 출력 품질을 안정화하는 전략. 재학습 없이 적용 가능.
5. **bitsandbytes**: HuggingFace transformers와 연동되는 INT8/INT4 양자화 라이브러리.

### 선행 지식
- **Python 프로그래밍**: 클래스, 함수, 라이브러리 사용 (중급)
- **PyTorch 모델링**: 텐서 연산, 모델 로딩, GPU 메모리 관리 (중급)
- **부동소수점 이해**: FP32, FP16, BF16의 비트 구조와 정밀도 차이 (기초)
- **선수 학습**: Chapter 5-1 PEFT 실습 완료 권장 (LoRA + Quantization 결합 이해)

---

## 실습 구성

### 학습 방향

- **실습 구성 방식**
  - 문제와 정답 코드가 병렬로 제공되며, 각 단계별 TODO 영역을 채우며 학습자가 직접 구현합니다.

- **Required Package**
  - `transformers>=4.55.0` - HuggingFace 모델 로딩 및 추론
  - `bitsandbytes>=0.43.0` - INT8/INT4 양자화 지원
  - `torch>=2.0.0` - PyTorch 딥러닝 프레임워크
  - `accelerate>=0.30.0` - 모델 분산 로딩 및 GPU 관리

- **Step 요약** (문제 체감 → 해결 구조)
  - **Step 1 (5분)**: 환경 설정 — 라이브러리 설치 및 GPU 확인
  - **Step 2 (10분)**: FP16 모델 로딩 시도 (문제 체감) — 메모리 사용량 측정
  - **Step 3 (10분)**: INT4 양자화 적용 — FP16과 성능 비교
  - **Step 4 (10분)**: 환경 변화 입력 품질 저하 (문제 체감) — 오타/노이즈 테스트
  - **Step 5 (10분)**: TTP 전략 적용 — 프롬프트 강화로 품질 회복

- **데이터셋 개요 및 저작권 정보**
  - **데이터셋 명**: 고정 입력 샘플 (직접 제작)
  - **데이터셋 개요**: 정상 입력 5개 + 환경 변화 입력 5개 (오타, 노이즈, 모호함, 조건 변화)
  - **사용 목적**: FP16/INT4 비교, TTP 효과 검증
  - **저작권/출처**: 실습용 직접 제작 (라이선스 제약 없음)
  - **주의사항**: 실제 서비스 데이터와 다를 수 있으며, 실험 결과는 참고용입니다

### 문제 설명

- **문제 개요**: 이 실습은 **"문제 체감 → 해결"** 흐름으로 진행됩니다. 학습자는 먼저 **FP16 모델의 높은 메모리 사용량을 체감**하고, **INT4 양자화로 경량화 문제를 해결**합니다. 이후 **환경 변화 입력(오타/노이즈)에서 품질 저하를 체감**하고, **TTP(Test-time Prompting) 전략으로 품질을 회복**하는 방법을 경험합니다.
- **요구사항 요약**
  - FP16 모델로 추론하고 latency/memory 측정
  - INT4 양자화 모델로 전환 후 성능 비교
  - 환경 변화 입력에서 품질 저하 관찰
  - TTP 템플릿 적용 후 품질 개선 확인

### 학습 문제

- **Step 2 — FP16 모델 로딩**
  - **TODO 1**: FP16 모델로 추론 수행 및 latency/memory 측정 *(연결: Quantization / 성능 지표 측정)*
  - **1줄 요약**: 기준 성능 지표 수집
- **Step 3 — INT4 양자화 적용**
  - **TODO 2**: BitsAndBytesConfig로 INT4 모델 로딩 및 FP16과 비교 *(연결: PTQ / bitsandbytes)*
  - **1줄 요약**: 양자화 효과 측정 (메모리 50~75% 절감 목표)
- **Step 5 — TTP 전략 적용**
  - **TODO 3**: TTP 템플릿 적용 후 품질 변화 비교 *(연결: Test-time Prompting)*
  - **1줄 요약**: 환경 변화 입력에서 품질 개선 확인

---

# Step 1: 환경 설정

라이브러리를 설치하고 Quantization의 개념을 이해합니다.

### Setup: 라이브러리 설치

In [ ]:
%%capture
# 로컬 환경용 설치 (RTX 5060ti, 16GB VRAM)
%pip install "transformers>=4.55.0" "bitsandbytes>=0.43.0"
%pip install "torch>=2.0.0" "accelerate>=0.30.0"

### Concept Check: 5-1에서 배운 QLoRA와의 연결

> **🔗 5-1 복습**: Chapter 5-1에서 우리는 **QLoRA**(Quantized LoRA)를 사용하여 LLM을 파인튜닝했습니다.
> 5-2에서는 **추론 시점의 양자화**를 다룹니다. 두 챕터가 어떻게 연결되는지 명확히 이해해봅시다!

**5-1에서 배운 것 (학습 시 양자화):**
- **목적**: 제한된 GPU 메모리로 대형 모델 학습
- **방식**: Base model을 4-bit(NF4) 양자화 + LoRA adapter(FP16) 학습
- **핵심 키워드**: QLoRA, NF4, LoRA adapter, Fine-tuning

**5-2에서 배울 것 (추론 시 양자화):**
- **목적**: 추론 속도 향상 및 메모리 절감
- **방식**: 학습된 모델 전체를 INT4로 양자화 (PTQ)
- **핵심 키워드**: PTQ, INT4, bitsandbytes, 추론 최적화

---

#### 핵심 차이점: 학습 vs 추론 양자화

| 구분 | 5-1: 학습 시 양자화 (QLoRA) | 5-2: 추론 시 양자화 (PTQ) |
|------|---------------------------|-------------------------|
| **시점** | Fine-tuning 중 | Fine-tuning 완료 후 |
| **목적** | 학습 가능하게 하기 | 추론 빠르게 하기 |
| **양자화 대상** | Base model만 (4-bit NF4) | 모델 전체 (INT4) |
| **학습 가능?** | LoRA adapter만 학습 (FP16) | ❌ 학습 불가 (추론 전용) |
| **정밀도** | NF4 (4-bit, 가중치 분포 최적화) | INT4 (4-bit, 정수 양자화) |
| **라이브러리** | Unsloth + bitsandbytes | bitsandbytes (load_in_4bit) |
| **메모리 절감** | 학습 가능하게 (8배 절감) | 추론 시 75% 절감 |

---

#### 전체 파이프라인에서의 위치

```
┌──────────────────────────────────────────────────────────────────┐
│                    LLM 활용 전체 파이프라인                        │
└──────────────────────────────────────────────────────────────────┘

1. [사전학습] 
   대규모 텍스트로 학습 → FP32/BF16 (높은 정밀도)

2. [파인튜닝 - Chapter 5-1 ⭐]
   도메인 데이터로 적응 → QLoRA (4-bit base + FP16 adapter)
   → 제한된 GPU로 학습 가능!

3. [추론 최적화 - Chapter 5-2 ⭐]
   실제 서비스 배포 → INT4 양자화 (PTQ)
   → 메모리 75% 절감 + 속도 향상!

4. [추론 품질 개선 - Chapter 5-2]
   환경 변화 대응 → TTP (Test-time Prompting)
   → 재학습 없이 품질 보정!
```

> **핵심**: 5-1(학습)과 5-2(추론)는 **별개의 단계**입니다. 
> - 5-1: "어떻게 학습할까?" → QLoRA로 학습
> - 5-2: "어떻게 빠르게 추론할까?" → INT4로 추론

---

#### QLoRA의 "Q"와 PTQ의 차이점

**용어 혼동 주의!** 
- **QLoRA의 Q**: Quantized LoRA → **학습 중** base model 양자화
- **PTQ의 Q**: Post-Training Quantization → **학습 후** 모델 양자화

| 항목 | QLoRA (5-1) | PTQ (5-2) |
|------|-------------|-----------|
| **Q의 의미** | Quantized (학습 시) | Post-Training (학습 후) |
| **언제?** | Fine-tuning 중 | Fine-tuning 완료 후 |
| **목적** | 학습 가능하게 | 추론 빠르게 |
| **gradient 계산?** | LoRA adapter만 계산 | ❌ 계산 안 함 |

### Concept Check: 비트 정밀도 통합 요약 (5-1 + 5-2)

5-1과 5-2에서 다양한 비트 정밀도가 등장합니다. 전체 그림을 한눈에 정리해봅시다.

---

#### 비트 정밀도 전체 맵

| 정밀도 | 비트 수 | 용도 | 주요 챕터 | 메모리 (1B 모델) | 학습 가능? |
|--------|---------|------|----------|----------------|----------|
| **FP32** | 32-bit | 사전학습, Optimizer | - | 4GB | ✅ |
| **FP16/BF16** | 16-bit | Mixed Precision 학습 | 5-1 (LoRA adapter) | 2GB | ✅ |
| **INT4/NF4** | 4-bit | 학습 베이스 (5-1) + 추론 최적화 (5-2) | **5-1, 5-2** | 0.5GB | ❌ (base만) |

> **💡 INT4 vs NF4**: NF4(Normal Float 4-bit)는 INT4의 LLM 최적화 버전입니다.
> - **INT4**: 4-bit 정수 양자화 (포괄적 용어)
> - **NF4**: 정규분포를 가정한 4-bit 양자화 (LLM 가중치 분포에 최적화)
> - 실습 코드에서는 `bnb_4bit_quant_type="nf4"`로 NF4를 사용합니다 (학습/추론 모두)

---

#### 각 챕터에서 사용하는 정밀도 조합

**Chapter 5-1: QLoRA 파인튜닝**
```
┌─────────────────────┐
│  Base Model (고정)   │ → NF4 (4-bit)     ❄️ Frozen
└─────────────────────┘
         +
┌─────────────────────┐
│ LoRA Adapter (학습)  │ → FP16 (16-bit)   🔥 Trainable
└─────────────────────┘

결과: 메모리 8배 절감 + 학습 가능
```

**Chapter 5-2: PTQ 추론 최적화**
```
┌─────────────────────┐
│  모델 전체 (고정)    │ → NF4 (4-bit)     ❄️ Inference Only
└─────────────────────┘

결과: 메모리 75% 절감 + 추론 속도 향상
```

---

#### 정밀도 선택 가이드

| 상황 | 권장 정밀도 | 이유 | 관련 챕터 |
|------|------------|------|----------|
| **학습 (충분한 GPU)** | FP16/BF16 | 속도와 정확도 균형 | - |
| **학습 (제한된 GPU)** | **NF4 base + FP16 adapter** | 8배 메모리 절감 | **5-1 (QLoRA)** |
| **추론 (메모리 중요)** | **NF4** | 메모리 75% 절감 | **5-2 (PTQ)** |
| **추론 (품질 중요)** | FP16 | 최소 손실 | - |

---

#### 메모리 비교: 실제 수치로 체감

**7B 모델 기준:**

| 단계 | 정밀도 조합 | 총 메모리 | 16GB VRAM? |
|------|-----------|----------|-----------|
| **사전학습** | FP32 | ~28GB | ❌ 불가능 |
| **Full FT** | FP16 (model+grad+optim) | ~50-70GB | ❌ 불가능 |
| **5-1 (QLoRA)** | NF4 base + FP16 LoRA | ~8-12GB | ✅ 가능! |
| **5-2 (PTQ 추론)** | NF4 전체 | ~3.5GB | ✅ 가능! |

> **핵심**: 5-1(학습)과 5-2(추론)는 **동일한 4-bit NF4 양자화**를 사용하지만, 
> 5-1은 학습 가능하게, 5-2는 추론 최적화에 초점을 맞춥니다!

---

이제 5-2 본격 시작! 아래에서 PTQ(Post-Training Quantization)를 깊이 있게 다뤄봅시다. 🚀


### Concept Check: Quantization이란?

> **Quantization(양자화)**은 모델의 가중치(weights)를 표현하는 비트 수를 줄여
> 메모리 사용량과 추론 속도를 개선하는 기법입니다.

**정밀도별 비교:**

| 정밀도 | 비트 수 | 메모리 (1B 파라미터 기준) | 특징 |
|--------|---------|---------------------------|------|
| FP32 | 32비트 | 4GB | 학습 기본 정밀도 |
| FP16 | 16비트 | 2GB | 추론 기본 정밀도 |
| INT8 | 8비트 | 1GB | 50% 메모리 절감, 속도 향상 |
| INT4 | 4비트 | 0.5GB | 75% 메모리 절감, 품질 저하 위험 |

**양자화가 필요한 이유:**
- 로컬 GPU (16GB VRAM)에서 대형 모델 사용 가능
- 추론 비용 절감 (클라우드 배포 시)
- 응답 속도 향상 (실시간 서비스)

**트레이드오프:**
- 메모리/속도 ↑ vs 정확도 ↓
- 특히 환경 변화(노이즈, 오타 등)에서 품질 저하가 더 심함

### Concept Check: 주요 양자화 방식 비교

> 양자화 방식은 다양하며, 각 방식마다 장단점이 있습니다. 이 실습에서는 **bitsandbytes (INT4)**를 사용하지만,
> 실무에서는 상황에 따라 적절한 방식을 선택해야 합니다.

**주요 양자화 방식:**

| 방식 | 특징 | 장점 | 단점 | 적합 환경 |
|------|------|------|------|----------|
| **bitsandbytes (INT4)** | 런타임 양자화, HuggingFace 연동 | 간편한 적용, 추가 변환 불필요 | 품질 저하 가능 | 빠른 실험, 프로토타이핑 |
| **GPTQ** | 사전 양자화 (INT4), 가중치 보정 | 높은 압축률, 빠른 추론 | 양자화 시간 소요 | 배포 최적화 |
| **AWQ** | 활성화 기반 양자화, 중요 가중치 보존 | 품질 유지 우수, INT4에서도 고품질 | 특정 모델만 지원 | 고품질 필요 서비스 |
| **GGUF** | llama.cpp 형식, CPU/GPU 유연 | 로컬 추론 최적화, 경량 | HuggingFace 직접 미지원 | 로컬 추론, 엣지 배포 |

**이 실습에서 bitsandbytes INT4를 선택한 이유:**
1. HuggingFace transformers와 자연스럽게 연동
2. 별도의 모델 변환 과정 없이 바로 적용 가능
3. 16GB VRAM에서 75% 메모리 절감 효과
4. 학습 목적에 적합한 간결한 코드

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import time

# GPU 확인
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu_stats.name}")
    print(f"최대 메모리: {gpu_stats.total_memory / 1024**3:.2f} GB")
else:
    print("GPU를 사용할 수 없습니다. CUDA 환경을 확인하세요.")

---

# Step 2: FP16 모델 로딩 시도 (문제 체감)

FP16 모델을 로딩하여 메모리 사용량을 직접 체감합니다.

### Concept Check: FP16 모델의 메모리 사용

**FP16 (Half Precision)**
- 16비트 부동소수점으로 모델 가중치 표현
- 1B 파라미터 모델 기준 약 2GB 메모리 필요
- Qwen2.5-1.5B 모델: 약 3~4GB VRAM 사용

**문제점:**
- 16GB VRAM에서 7B 이상 모델은 로딩 어려움
- 배치 처리 시 추가 메모리 필요
- 더 큰 모델(13B, 70B)은 사실상 불가능

### Guided Build: FP16 모델 로딩

In [ ]:
# 사용할 모델 (경량 모델 선택 - Gated가 아닌 모델)
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# FP16 모델 로딩
print("FP16 모델 로딩 중...")
torch.cuda.reset_peak_memory_stats()  # 메모리 통계 초기화
model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 모델 로딩 직후 메모리 측정 (가중치 메모리)
model_memory_fp16 = torch.cuda.memory_allocated() / 1024**3
print(f"모델 로딩 완료: {model_name}")
print(f"FP16 모델 메모리: {model_memory_fp16:.2f} GB")

### Guided Build: 모델 파라미터 탐색 및 메모리 계산

모델을 로딩했으니, 실제로 파라미터가 몇 개인지, 메모리가 얼마나 필요한지 직접 확인해봅시다.
이 과정을 통해 양자화가 왜 필요한지 체감할 수 있습니다.

In [ ]:
# 모델 파라미터 수 확인
total_params = sum(p.numel() for p in model_fp16.parameters())
trainable_params = sum(p.numel() for p in model_fp16.parameters() if p.requires_grad)

print("=" * 50)
print("[모델 파라미터 분석]")
print("=" * 50)
print(f"총 파라미터 수: {total_params:,}")
print(f"학습 가능한 파라미터: {trainable_params:,}")

# 메모리 계산: 파라미터 수 × 바이트
memory_fp32 = total_params * 4 / 1024**3  # 32비트 = 4바이트
memory_fp16 = total_params * 2 / 1024**3  # 16비트 = 2바이트
memory_int4 = total_params * 0.5 / 1024**3  # 4비트 = 0.5바이트

print(f"\n[이론적 메모리 사용량 계산]")
print(f"FP32 (32-bit): {memory_fp32:.2f} GB")
print(f"FP16 (16-bit): {memory_fp16:.2f} GB")
print(f"INT4 (4-bit):  {memory_int4:.2f} GB")

print(f"\n[메모리 절감 효과]")
print(f"FP32 → FP16: {(1 - memory_fp16/memory_fp32)*100:.0f}% 절감")
print(f"FP16 → INT4: {(1 - memory_int4/memory_fp16)*100:.0f}% 절감")
print(f"FP32 → INT4: {(1 - memory_int4/memory_fp32)*100:.0f}% 절감")

### Guided Build: FP16 vs FP32 숫자 표현 비교

양자화로 인한 정밀도 손실을 직접 체험해봅시다. 
같은 숫자를 FP32와 FP16으로 표현했을 때 어떤 차이가 나는지 확인합니다.

In [ ]:
# FP32 vs FP16 숫자 표현 비교 체험
print("=" * 50)
print("[FP32 vs FP16 숫자 표현 비교]")
print("=" * 50)

# 테스트할 값들
test_values = [
    3.141592653589793,   # 원주율 (정밀도 테스트)
    0.00001,             # 아주 작은 수
    65504.0,             # FP16 최대값 근처
    0.123456789,         # 일반적인 소수
]

for value in test_values:
    fp32 = torch.tensor(value, dtype=torch.float32)
    fp16 = torch.tensor(value, dtype=torch.float16)
    
    print(f"\n원본값: {value}")
    print(f"FP32:   {fp32.item():.15f}")
    print(f"FP16:   {fp16.item():.15f}")
    print(f"오차:   {abs(value - fp16.item()):.15f}")

print("\n" + "=" * 50)
print("💡 관찰: FP16은 정밀도가 낮아 작은 오차가 발생합니다.")
print("   이 오차가 모델 전체에 누적되면 품질 저하로 이어질 수 있습니다.")

### Guided Build: model.generate()로 추론 수행하기

TODO 1을 수행하기 전에, `model.generate()` API 사용법을 간단히 살펴봅시다.

```python
# 기본 추론 예시
with torch.no_grad():
    outputs = model.generate(
        inputs,                  # 토큰화된 입력
        max_new_tokens=50,       # 생성할 최대 토큰 수
        do_sample=True,          # 샘플링 사용 여부
        temperature=0.7,         # 샘플링 온도 (높을수록 다양)
    )
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
```

**시간 측정 패턴:**
```python
start_time = time.time()
# ... 측정할 코드 ...
latency = time.time() - start_time
```

**메모리 측정 패턴:**
```python
torch.cuda.synchronize()  # GPU 연산 완료 대기
memory = torch.cuda.max_memory_allocated() / 1024**3  # GB 단위
```

### TODO 1: FP16 모델로 추론 수행 및 latency/memory 측정

- **요구사항**: FP16 모델로 추론을 수행하고 latency(추론 시간)와 memory(메모리 사용량)를 측정하세요.
- **입력**: 테스트 프롬프트 ("서울에서 부산까지 KTX로 얼마나 걸리나요?")
- **출력**: 모델 응답, latency (초), memory (GB)
- **힌트**: `time.time()`으로 시간 측정, `torch.cuda.memory_allocated()`로 메모리 측정

In [ ]:
# TODO: FP16 모델로 추론 수행 및 latency/memory 측정

def measure_inference(model, tokenizer, prompt, max_new_tokens=128):
    """모델 추론을 수행하고 latency와 memory를 측정합니다."""
    
    # 메모리 측정 (추론 전)
    torch.cuda.synchronize()
    memory_before = torch.cuda.memory_allocated() / 1024**3
    
    # 입력 토큰화
    messages = [
        {"role": "user", "content": prompt}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True
    )
    input_ids = inputs["input_ids"].to(model.device)
    input_length = input_ids.shape[1]
    
    # 재현성을 위한 seed 설정
    torch.manual_seed(42)
    
    # 추론 시간 측정
    start_time = time.time()
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    torch.cuda.synchronize()
    end_time = time.time()
    
    # 메모리 측정 (추론 후)
    memory_after = torch.cuda.max_memory_allocated() / 1024**3
    
    # 결과 디코딩
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    latency = end_time - start_time
    
    return response, latency, memory_after

# 테스트 프롬프트
test_prompt = "서울에서 부산까지 KTX로 얼마나 걸리나요?"

# FP16 추론 실행
response_fp16, latency_fp16, memory_fp16 = measure_inference(model_fp16, tokenizer, test_prompt)

print("=" * 50)
print("[FP16 모델 추론 결과]")
print("=" * 50)
print(f"입력: {test_prompt}")
print(f"응답: {response_fp16}")
print(f"\nLatency: {latency_fp16:.2f}초")
print(f"Memory: {memory_fp16:.2f} GB")
print("\n⚠️ 체감: 메모리를 많이 사용하네! 더 큰 모델은 16GB에 못 올리겠다.")

### Test: FP16 메모리 사용량 확인

In [ ]:
# FP16 결과 저장 (나중에 INT4와 비교용)
fp16_results = {
    "latency": latency_fp16,
    "memory": memory_fp16,
    "model_memory": model_memory_fp16,  # 모델 가중치 메모리
    "response": response_fp16
}

print(f"FP16 모델 가중치 메모리: {model_memory_fp16:.2f} GB")
print(f"FP16 추론 피크 메모리: {memory_fp16:.2f} GB")
print(f"16GB VRAM의 {(memory_fp16/16)*100:.1f}% 사용 중")

# FP16 모델 메모리 해제 (INT4 로딩을 위해)
del model_fp16
torch.cuda.empty_cache()
print("\nFP16 모델 메모리 해제 완료")

---

# Step 3: INT4 양자화 적용

bitsandbytes를 사용하여 INT4 양자화를 적용하고 FP16과 비교합니다.

### Concept Check: PTQ(Post-Training Quantization)

**PTQ(Post-Training Quantization)**
- 이미 학습된 모델에 양자화를 적용하는 방식
- 재학습 없이 빠르게 적용 가능
- 정확도 손실이 있을 수 있지만, 대부분의 경우 허용 가능한 수준

**bitsandbytes 라이브러리**
- HuggingFace transformers와 연동되는 양자화 라이브러리
- `load_in_8bit=True` 옵션으로 INT8 적용
- `load_in_4bit=True` 옵션으로 INT4 적용 (더 공격적인 압축, **이 실습에서 사용**)

**BitsAndBytesConfig (INT4)**
```python
BitsAndBytesConfig(
    load_in_4bit=True,                    # INT4 양자화
    bnb_4bit_compute_dtype=torch.float16, # 연산은 FP16
    bnb_4bit_quant_type="nf4",            # NF4 양자화 타입
    bnb_4bit_use_double_quant=True,       # 이중 양자화
)
```

**INT8 vs INT4 선택 기준:**
| 방식 | 메모리 절감 | 속도 | 품질 | 적합한 상황 |
|------|-----------|------|------|-----------|
| INT8 | 50% | 오버헤드 있음 | 높음 | 품질 중시 |
| **INT4** | **75%** | **빠름** | 중간 | **메모리 제한 환경** |

### Guided Build: 양자화 과정 단계별 체험

양자화가 내부적으로 어떻게 작동하는지 간단한 텐서로 직접 체험해봅시다.

**양자화 수식:**
```
Q(x) = round(x / scale) + zero_point
```

- **scale**: 원본 범위를 INT8 범위(-128~127)에 맞추는 스케일
- **zero_point**: 0이 매핑되는 정수 값

In [ ]:
# 양자화 과정 단계별 체험
print("=" * 70)
print("[양자화 과정 단계별 체험: INT8 vs INT4 비교]")
print("=" * 70)

# 1. 원본 텐서 (FP16 가중치를 시뮬레이션)
original = torch.tensor([0.5, 1.2, -0.3, 2.1, -1.5], dtype=torch.float16)
print(f"\n1️⃣ 원본 텐서 (FP16):")
print(f"   {original.tolist()}")

# 2. 양자화 파라미터 계산 (공통)
x_min, x_max = original.min().item(), original.max().item()
print(f"\n2️⃣ 데이터 범위 분석:")
print(f"   x_min: {x_min:.4f}, x_max: {x_max:.4f}")

# ============================================================
# INT8 양자화 (8비트 = 256 레벨)
# ============================================================
print("\n" + "=" * 70)
print("[INT8 양자화 (8-bit, 256 레벨)]")
print("=" * 70)

scale_int8 = (x_max - x_min) / 255  # 0~255 범위
zero_point_int8 = round(-x_min / scale_int8)

print(f"   scale: {scale_int8:.6f}")
print(f"   zero_point: {zero_point_int8}")

# INT8 양자화
quantized_int8 = torch.round(original / scale_int8 + zero_point_int8).to(torch.uint8)
print(f"\n   양자화된 값 (INT8): {quantized_int8.tolist()}")

# INT8 역양자화
dequantized_int8 = (quantized_int8.float() - zero_point_int8) * scale_int8
print(f"   역양자화된 값:     {[round(x, 4) for x in dequantized_int8.tolist()]}")

# INT8 오차
error_int8 = (original.float() - dequantized_int8).abs()
print(f"\n   양자화 오차:       {[round(x, 6) for x in error_int8.tolist()]}")
print(f"   평균 오차: {error_int8.mean().item():.6f}")

# ============================================================
# INT4 양자화 (4비트 = 16 레벨)
# ============================================================
print("\n" + "=" * 70)
print("[INT4 양자화 (4-bit, 16 레벨)]")
print("=" * 70)

scale_int4 = (x_max - x_min) / 15  # 0~15 범위 (4비트)
zero_point_int4 = round(-x_min / scale_int4)

print(f"   scale: {scale_int4:.6f}")
print(f"   zero_point: {zero_point_int4}")

# INT4 양자화 (0~15 범위로 클램핑)
quantized_int4 = torch.round(original / scale_int4 + zero_point_int4).clamp(0, 15).to(torch.uint8)
print(f"\n   양자화된 값 (INT4): {quantized_int4.tolist()}")

# INT4 역양자화
dequantized_int4 = (quantized_int4.float() - zero_point_int4) * scale_int4
print(f"   역양자화된 값:     {[round(x, 4) for x in dequantized_int4.tolist()]}")

# INT4 오차
error_int4 = (original.float() - dequantized_int4).abs()
print(f"\n   양자화 오차:       {[round(x, 6) for x in error_int4.tolist()]}")
print(f"   평균 오차: {error_int4.mean().item():.6f}")

# ============================================================
# INT8 vs INT4 비교 요약
# ============================================================
print("\n" + "=" * 70)
print("[INT8 vs INT4 비교 요약]")
print("=" * 70)
print(f"{'항목':<20} {'INT8 (8-bit)':>15} {'INT4 (4-bit)':>15}")
print("-" * 70)
print(f"{'표현 가능 레벨':<20} {'256':>15} {'16':>15}")
print(f"{'메모리 (1B 파라미터)':<20} {'1 GB':>15} {'0.5 GB':>15}")
print(f"{'평균 양자화 오차':<20} {error_int8.mean().item():>15.6f} {error_int4.mean().item():>15.6f}")
print(f"{'오차 증가율':<20} {'기준':>15} {f'{(error_int4.mean() / error_int8.mean()):.1f}x':>15}")

print("\n" + "=" * 70)
print("💡 관찰:")
print("   - INT4는 INT8 대비 메모리 50% 추가 절감 (8bit → 4bit)")
print("   - 하지만 표현 가능한 레벨이 256 → 16으로 감소")
print(f"   - 양자화 오차가 약 {(error_int4.mean() / error_int8.mean()):.1f}배 증가")
print("   - 트레이드오프: 메모리 절감 ↑ vs 정밀도 손실 ↑")
print("=" * 70)

### Concept Check: BitsAndBytesConfig 파라미터 상세 (INT4)

bitsandbytes의 INT4 양자화는 **NF4(Normal Float 4-bit)** 알고리즘을 사용합니다.
주요 파라미터를 이해하면 양자화 품질을 조절할 수 있습니다.

```python
BitsAndBytesConfig(
    load_in_4bit=True,                    # INT4 양자화 활성화
    bnb_4bit_compute_dtype=torch.float16, # 연산 시 사용할 dtype
    bnb_4bit_quant_type="nf4",            # 양자화 타입
    bnb_4bit_use_double_quant=True,       # 이중 양자화
)
```

**주요 파라미터:**

| 파라미터 | 기본값 | 역할 |
|----------|--------|------|
| `bnb_4bit_compute_dtype` | float32 | 연산 시 사용할 dtype (float16 권장) |
| `bnb_4bit_quant_type` | "fp4" | 양자화 타입 ("nf4" 또는 "fp4") |
| `bnb_4bit_use_double_quant` | False | 이중 양자화 (추가 메모리 절감) |

**NF4 vs FP4:**
- **NF4 (Normal Float 4)**: 정규분포 가정, LLM 가중치에 최적화
- **FP4 (Float Point 4)**: 균등 분포, 일반적인 양자화

**이중 양자화 (Double Quantization):**
- 양자화 상수(scale) 자체도 양자화하여 추가 메모리 절감
- 약 0.4bit/param 추가 절감 효과

**왜 INT4가 INT8보다 빠를까?**
- INT8 (LLM.int8()): 런타임에 이상치 분리 → 오버헤드 발생
- INT4 (NF4): 사전에 양자화 완료, FP16 연산 → 오버헤드 적음

> **팁**: 품질이 중요하면 `bnb_4bit_quant_type="nf4"`, 메모리가 극도로 제한되면 `bnb_4bit_use_double_quant=True`

### Guided Build: INT4 양자화 모델 로딩 예시

TODO 2를 수행하기 전에, `BitsAndBytesConfig`와 `from_pretrained`를 함께 사용하는 완전한 예시를 살펴봅시다.

```python
# 1. BitsAndBytesConfig로 INT4 양자화 설정 생성
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # INT4 양자화 활성화
    bnb_4bit_compute_dtype=torch.float16, # 연산은 FP16으로 수행
    bnb_4bit_quant_type="nf4",            # NF4 양자화 타입
    bnb_4bit_use_double_quant=True,       # 이중 양자화
)

# 2. from_pretrained에 quantization_config 전달
model = AutoModelForCausalLM.from_pretrained(
    "model_name",
    quantization_config=quantization_config,  # 여기에 전달!
    device_map="auto",
)
```

**핵심 포인트:**
- `BitsAndBytesConfig`는 양자화 "설정"만 정의
- 실제 양자화는 `from_pretrained`에서 발생
- `device_map="auto"`로 GPU에 자동 배치
- `bnb_4bit_compute_dtype=torch.float16`으로 연산 속도 향상

### TODO 2: INT4 양자화 모델 로딩 및 FP16과 비교

- **요구사항**: BitsAndBytesConfig를 사용하여 INT4 양자화 모델을 로딩하고, FP16과 latency/memory를 비교하세요.
- **입력**: 동일한 테스트 프롬프트
- **출력**: FP16/INT4 latency, memory, 출력 품질 비교 표
- **힌트**: `load_in_4bit=True` 옵션과 `bnb_4bit_compute_dtype=torch.float16` 사용

In [ ]:
# TODO: INT4 양자화 모델 로딩 및 FP16과 비교

# INT4 양자화 설정 (NF4 - Normal Float 4-bit)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,                          # INT4 양자화 활성화
    bnb_4bit_compute_dtype=torch.float16,       # 연산은 FP16으로 수행
    bnb_4bit_quant_type="nf4",                  # NF4 양자화 타입
    bnb_4bit_use_double_quant=True,             # 이중 양자화로 추가 메모리 절감
)

# INT4 모델 로딩
print("INT4 양자화 모델 로딩 중...")
torch.cuda.reset_peak_memory_stats()  # 메모리 통계 초기화
model_int4 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
)

# 모델 로딩 직후 메모리 측정 (가중치 메모리)
model_memory_int4 = torch.cuda.memory_allocated() / 1024**3
print(f"INT4 모델 로딩 완료: {model_name}")
print(f"INT4 모델 메모리: {model_memory_int4:.2f} GB")

In [ ]:
# INT4 추론 실행
response_int4, latency_int4, memory_int4 = measure_inference(model_int4, tokenizer, test_prompt)

print("=" * 50)
print("[INT4 모델 추론 결과]")
print("=" * 50)
print(f"입력: {test_prompt}")
print(f"응답: {response_int4}")
print(f"\nLatency: {latency_int4:.2f}초")
print(f"추론 피크 메모리: {memory_int4:.2f} GB")

In [ ]:
# FP16 vs INT4 비교
print("\n" + "=" * 70)
print("[FP16 vs INT4 비교]")
print("=" * 70)
print(f"{'지표':<20} {'FP16':>15} {'INT4':>15} {'변화율':>15}")
print("-" * 70)

# 모델 가중치 메모리 비교 (핵심 지표)
model_memory_change = ((model_memory_int4 - fp16_results['model_memory']) / fp16_results['model_memory']) * 100
print(f"{'모델 메모리 (GB)':<20} {fp16_results['model_memory']:>15.2f} {model_memory_int4:>15.2f} {model_memory_change:>14.1f}%")

# Latency 비교
latency_change = ((latency_int4 - fp16_results['latency']) / fp16_results['latency']) * 100
print(f"{'Latency (초)':<20} {fp16_results['latency']:>15.2f} {latency_int4:>15.2f} {latency_change:>14.1f}%")

# 추론 피크 메모리 비교
peak_memory_change = ((memory_int4 - fp16_results['memory']) / fp16_results['memory']) * 100
print(f"{'피크 메모리 (GB)':<20} {fp16_results['memory']:>15.2f} {memory_int4:>15.2f} {peak_memory_change:>14.1f}%")

print("\n✅ 체감: INT4 양자화로 모델 메모리가 대폭 줄었다!")
print("   → 같은 GPU에서 더 큰 모델을 로딩할 수 있게 됨")

### Test: INT4 메모리 절감 확인

In [ ]:
# INT4 결과 저장
int4_results = {
    "latency": latency_int4,
    "memory": memory_int4,
    "model_memory": model_memory_int4,
    "response": response_int4
}

# 메모리 절감 확인 (모델 가중치 기준)
memory_saved = fp16_results['model_memory'] - model_memory_int4
memory_saved_percent = (memory_saved / fp16_results['model_memory']) * 100

print(f"모델 메모리 절감량: {memory_saved:.2f} GB ({memory_saved_percent:.1f}%)")

if memory_saved_percent >= 50:
    print("✅ INT4 양자화로 모델 메모리를 50% 이상 절감했습니다!")
    print("   이론적으로 FP16 → INT4는 75% 절감 (16bit → 4bit)")
else:
    print("⚠️ 메모리 절감 효과가 예상보다 적습니다.")
    print("   (이중 양자화 메타데이터 등의 오버헤드 존재)")

### Concept Check: 모델 메모리 vs 피크 메모리 — 왜 차이가 날까?

위 비교 결과에서 **모델 메모리**와 **피크 메모리**의 차이를 이해하는 것이 중요합니다.

---

#### 두 지표의 의미

| 지표 | 측정 시점 | 포함 내용 |
|------|----------|----------|
| **모델 메모리** | 모델 로딩 직후 | 모델 가중치(weights, biases)만 |
| **피크 메모리** | 추론 중 최대값 | 모델 가중치 + 활성화값 + KV 캐시 + 임시 버퍼 |

---

#### 왜 INT4에서 피크 메모리 절감이 작을까?

```
FP16:  모델 3.95GB → 피크 3.97GB  (차이: ~0.02GB)
INT4:  모델 1.73GB → 피크 3.83GB  (차이: ~2.10GB)
```

**핵심 포인트:**
- **양자화는 가중치(weights)만 압축**합니다
- 추론 시 **활성화값(activations)과 KV 캐시는 여전히 FP16/FP32**로 유지
- 따라서 INT4 모델도 추론 시 추가 메모리가 필요

---

#### 모델 크기에 따른 양자화 효과

| 모델 크기 | 가중치 비중 | 활성화 비중 | 양자화 효과 |
|-----------|-----------|-----------|------------|
| **1.5B (작음)** | 상대적 작음 | 상대적 큼 | 피크 메모리 절감 **적음** |
| **7B (중간)** | 중간 | 중간 | 피크 메모리 절감 **중간** |
| **70B (큼)** | 대부분 | 상대적 작음 | 피크 메모리 절감 **큼** |

**예시 (대략적 수치):**
```
1.5B 모델:
  - FP16 피크: ~4GB (가중치 3GB + 활성화 1GB)
  - INT4 피크: ~2GB (가중치 0.75GB + 활성화 1GB)
  - 절감률: ~50%

70B 모델:
  - FP16 피크: ~150GB (가중치 140GB + 활성화 10GB)
  - INT4 피크: ~45GB (가중치 35GB + 활성화 10GB)
  - 절감률: ~70%
```

---

#### 핵심 정리

> **작은 모델에서는** 활성화/KV캐시가 전체 메모리에서 차지하는 비중이 커서 양자화 효과가 피크 메모리에서 덜 체감됩니다.
>
> **큰 모델에서는** 가중치가 메모리의 대부분을 차지하므로 양자화의 피크 메모리 절감 효과가 극대화됩니다.
>
> 이것이 **70B+ 대형 모델에서 INT4 양자화가 필수적인 이유**입니다. (140GB → 45GB로 소비자 GPU에서 실행 가능)

---

# Step 4: 환경 변화 입력 품질 저하 (문제 체감)

환경 변화 입력(오타, 노이즈, 모호함)에서 INT4 모델의 품질 저하를 관찰합니다.

### Concept Check: 환경 변화란?

**환경 변화(Distribution Shift)**
- 실제 서비스에서 사용자 입력은 학습 데이터와 다를 수 있음
- 오타, 노이즈, 모호한 표현, 예상 외 질문 등

**양자화 모델에서 품질 저하가 심한 이유**
- 양자화로 인한 정밀도 손실이 누적됨
- 경계 케이스(edge case)에서 특히 취약
- 입력이 불명확할수록 모델의 "추론 여유"가 줄어듦

**환경 변화 유형:**
| 유형 | 예시 | 특징 |
|------|------|------|
| 정상 | "서울에서 부산까지 KTX로 얼마나 걸리나요?" | 명확한 질문 |
| 오타 | "서울에셔 부산까지 KTX로 얼마나 걸리나요?" | 오타 포함 |
| 노이즈 | "서울에서 부산까지... 음... KTX로 얼마나 걸리나요? 대략?" | 불필요한 표현 |
| 모호함 | "그거 얼마나 걸려?" | 맥락 부족 |
| 조건변화 | "밤 11시에 출발하면 다음날 아침에 도착할 수 있나요?" | 복잡한 조건 |

### Guided Build: 환경 변화 입력 샘플 정의

In [ ]:
# 환경 변화 입력 샘플 정의
test_samples = [
    {
        "type": "정상",
        "prompt": "서울에서 부산까지 KTX로 얼마나 걸리나요?",
        "expected_quality": "높음"
    },
    {
        "type": "오타",
        "prompt": "서울에셔 부산까지 KTX로 얼마나 걸리나요?",
        "expected_quality": "중간"
    },
    {
        "type": "노이즈",
        "prompt": "서울에서 부산까지... 음... KTX로 얼마나 걸리나요? 대략?",
        "expected_quality": "중간"
    },
    {
        "type": "모호함",
        "prompt": "그거 얼마나 걸려?",
        "expected_quality": "낮음"
    },
    {
        "type": "조건변화",
        "prompt": "밤 11시에 출발하면 다음날 아침에 도착할 수 있나요?",
        "expected_quality": "중간"
    }
]

print(f"총 {len(test_samples)}개의 테스트 샘플 정의 완료")

### Guided Build: INT4 모델에서 환경 변화 입력 테스트

In [ ]:
# 환경 변화 입력 테스트
env_change_results = []

print("=" * 70)
print("[INT4 모델 - 환경 변화 입력 테스트]")
print("=" * 70)

for sample in test_samples:
    response, latency, _ = measure_inference(model_int4, tokenizer, sample["prompt"], max_new_tokens=100)
    
    env_change_results.append({
        "type": sample["type"],
        "prompt": sample["prompt"],
        "response": response,
        "latency": latency
    })
    
    print(f"\n[{sample['type']}]")
    print(f"입력: {sample['prompt']}")
    print(f"응답: {response[:200]}..." if len(response) > 200 else f"응답: {response}")
    print("-" * 70)

In [ ]:
# 품질 저하 관찰
print("\n" + "=" * 50)
print("⚠️ 체감: 환경 변화 입력에서 품질이 저하되는 것 관찰")
print("=" * 50)
print("- 오타: 오타를 이해하지 못하거나 이상한 응답")
print("- 노이즈: 불필요한 표현에 혼란")
print("- 모호함: 맥락 부족으로 엉뚱한 응답")
print("- 조건변화: 복잡한 조건 처리 실패")
print("\n→ 이 문제를 해결하기 위해 TTP(Test-time Prompting) 전략을 적용해보자!")

### Concept Check: 왜 양자화 모델은 환경 변화에 취약할까?

위에서 관찰한 품질 저하의 근본 원인을 이해해봅시다.

---

#### 1. 정밀도 손실 누적

양자화는 각 가중치에서 작은 오차를 발생시킵니다. 
이 오차가 모델 전체(수십억 개의 파라미터)에 누적되면 출력에 영향을 미칩니다.

```
FP16 가중치: [0.123456, 0.234567, 0.345678, ...]
                 ↓ 양자화
양자화된 가중치: [0.125000, 0.234375, 0.343750, ...]  ← 작은 오차들
                 ↓ 수십억 번 연산
최종 출력: 오차 누적으로 다른 결과 가능
```

---

#### 2. Attention Score 계산의 미세한 차이

LLM의 핵심인 **Self-Attention**은 토큰 간 관계를 계산합니다.
양자화로 인해 attention score의 미세한 차이가 발생하면, **다른 토큰에 집중**하게 됩니다.

```
입력: "서울에셔 부산까지" (오타 포함)
           ↑
FP16: "에셔"를 "에서"와 유사하게 인식 (높은 attention)
양자화 모델: 미세한 차이로 "에셔"를 다르게 인식 (낮은 attention)
```

---

#### 3. 경계 케이스(Edge Case) 처리 능력 저하

학습 데이터에서 자주 본 패턴 → 강한 가중치 → 양자화에도 견딤
학습 데이터에서 드문 패턴 → 약한 가중치 → 양자화에 취약

| 입력 유형 | 학습 빈도 | 가중치 강도 | 양자화 영향 |
|----------|----------|-----------|-----------|
| 정상 문장 | 높음 | 강함 | 적음 |
| 오타/노이즈 | 낮음 | 약함 | **큼** |
| 모호한 표현 | 매우 낮음 | 매우 약함 | **매우 큼** |

---

#### 비유: 고화질 vs 저화질 이미지

```
FP16 = 고화질 사진 (4K)
  → 작은 글씨도 선명하게 보임
  → 세부 디테일 구분 가능

양자화 = 압축된 사진 (480p)
  → 대략적인 형태만 보임
  → 작은 글씨/디테일 뭉개짐

오타/노이즈 = 이미지의 작은 글씨
  → 고화질에서는 읽을 수 있음
  → 저화질에서는 읽기 어려움
```

---

#### 핵심 정리

1. **정밀도 손실**: 작은 오차가 누적되어 출력에 영향
2. **Attention 변화**: 미세한 가중치 차이로 다른 토큰에 집중
3. **경계 케이스 취약**: 드문 패턴일수록 양자화에 민감

> **결론**: 양자화된 모델은 "일반적인 입력"에서는 잘 작동하지만,
> "비정상적인 입력(오타, 노이즈, 모호함)"에서는 품질이 저하됩니다.
> 이를 보완하기 위해 **TTP(Test-time Prompting)**를 사용합니다!

---

# Step 5: TTP 전략 적용

Test-time Prompting을 적용하여 환경 변화 입력에서의 품질 저하를 완화합니다.

### Concept Check: Test-time Prompting(TTP)이란?

**Test-time Prompting (TTP)**
- 추론 시점에 프롬프트를 강화하여 모델 출력 품질을 안정화하는 전략
- 모델 재학습 없이 적용 가능
- 양자화로 인한 품질 저하를 프롬프트 엔지니어링으로 보완

**TTP 전략 유형:**

| 전략 | 설명 | 예시 |
|------|------|------|
| TTP-A | 출력 형식/제약 강화 | "정확한 소요 시간을 시간 단위로 답하세요." |
| TTP-B | few-shot 예시 포함 | 예시 1개 + 질문 형식 |

**TTP가 효과적인 이유:**
- 명확한 지시로 모델의 "추론 방향" 제시
- few-shot 예시로 원하는 출력 형식 학습
- 환경 변화 입력에서도 안정적인 응답 유도

### Concept Check: TTP가 왜 효과적일까?

TTP(Test-time Prompting)가 양자화된 모델의 품질을 개선하는 원리를 이해해봅시다.

---

#### 1. 명확한 지시 = 추론 방향 제시

양자화된 모델은 정밀도 손실로 인해 **"추론 여유"**가 줄어든 상태입니다.
입력이 불명확하면 여러 방향으로 추론할 수 있는데, 양자화 모델은 잘못된 방향을 선택할 확률이 높습니다.

```
불명확한 입력: "그거 얼마나 걸려?"
    ↓
FP16: 여러 해석 중 맥락에 맞는 것 선택 (추론 여유 있음)
양자화 모델: 잘못된 해석 선택 가능 (추론 여유 부족)
    ↓
TTP 적용: "교통 관련 질문입니다. 소요 시간을 답하세요."
    ↓
양자화 모델 + TTP: 명확한 방향 제시로 올바른 추론 유도
```

---

#### 2. Few-shot 예시 = 출력 형식 고정

Few-shot 예시를 제공하면 모델은 **"아, 이런 식으로 답하면 되는구나"**를 학습합니다.
양자화로 인한 불확실성을 예시로 보완하는 효과가 있습니다.

```
예시 제공 전:
  Q: "서울에셔 부산까지" → 다양한 형식으로 응답 가능

예시 제공 후:
  예시: "서울역에서 대전역까지 KTX로 약 50분~1시간 소요됩니다."
  Q: "서울에셔 부산까지" → 예시와 유사한 형식으로 응답
```

---

#### 3. 재학습 없이 적용 가능

TTP의 가장 큰 장점은 **모델 가중치를 변경하지 않는다**는 점입니다.

| 품질 개선 방법 | 모델 변경? | 시간 | 비용 | 유연성 |
|--------------|----------|------|------|--------|
| Fine-tuning | ✅ 변경 | 시간~일 | 높음 | 낮음 |
| RLHF | ✅ 변경 | 일~주 | 매우 높음 | 낮음 |
| **TTP** | ❌ 변경 없음 | 즉시 | 없음 | **높음** |

---

#### TTP 효과 요약

| 원리 | 효과 |
|------|------|
| 명확한 지시 | 잘못된 추론 방향 방지 |
| Few-shot 예시 | 출력 형식 고정 |
| 맥락 제공 | 모호함 해소 |
| 규칙 명시 | 일관된 응답 유도 |

> **핵심**: TTP는 모델의 "지식"을 변경하지 않고, "추론 방향"을 안내합니다.
> 양자화로 약해진 추론 능력을 프롬프트로 보조하는 전략입니다.

### Concept Check: 효과적인 TTP 템플릿 설계 가이드

실무에서 TTP를 적용할 때 참고할 수 있는 설계 원칙입니다.

---

#### 1. 규칙을 명시하라

모델이 따라야 할 규칙을 명확하게 작성합니다.

```
❌ Bad: "질문에 답해주세요."
✅ Good: "다음 규칙을 따라 답해주세요:
         1. 오타가 있어도 의도를 파악하세요.
         2. 간결하게 핵심만 답하세요.
         3. 불확실하면 '잘 모르겠습니다'라고 답하세요."
```

---

#### 2. 출력 형식을 지정하라

원하는 응답 형식을 구체적으로 지정합니다.

```
❌ Bad: "시간을 알려주세요."
✅ Good: "소요 시간을 '약 X시간 Y분' 형식으로 답하세요."

❌ Bad: "장단점을 설명해주세요."
✅ Good: "다음 형식으로 답하세요:
         [장점] ...
         [단점] ...
         [결론] ..."
```

---

#### 3. Few-shot 예시를 포함하라

1-2개의 예시만으로도 효과적입니다. 예시는 실제 사용 사례와 유사하게 작성합니다.

```
✅ Good:
"**예시:**
Q: 서울역에서 대전역까지 KTX로 얼마나 걸려요?
A: 서울역에서 대전역까지 KTX로 약 50분~1시간 정도 소요됩니다.

**질문:** {user_question}
**답변:**"
```

---

#### 4. 맥락을 제공하라

모델이 질문의 맥락을 이해하도록 도와줍니다.

```
❌ Bad: "얼마나 걸려?"
✅ Good: "당신은 한국 교통 전문가입니다. 
         사용자가 교통 관련 질문을 합니다.
         질문: 얼마나 걸려?"
```

---

#### TTP 템플릿 체크리스트

| 항목 | 확인 |
|------|------|
| 규칙이 명확하게 작성되었는가? | ☐ |
| 출력 형식이 지정되었는가? | ☐ |
| 예시가 포함되었는가? (선택) | ☐ |
| 맥락이 제공되었는가? | ☐ |
| 오타/노이즈 처리 규칙이 있는가? | ☐ |

---

#### 실무 팁

1. **점진적으로 개선**: 간단한 TTP부터 시작하고, 결과를 보며 규칙을 추가
2. **도메인 특화**: 사용 도메인(교통, 의료, 금융 등)에 맞는 예시와 규칙 사용
3. **A/B 테스트**: 여러 TTP 버전을 비교하여 최적의 템플릿 선택
4. **토큰 효율**: TTP가 너무 길면 토큰 비용 증가, 핵심만 포함

### Guided Build: TTP 템플릿 정의

In [ ]:
# TTP 템플릿 정의

# TTP-A: 출력 형식/제약 강화
TTP_A_TEMPLATE = """다음 질문에 답해주세요. 

**규칙:**
1. 오타나 불명확한 표현이 있어도 의도를 파악하세요.
2. 교통 관련 질문은 대략적인 소요 시간(시간 단위)으로 답하세요.
3. 맥락이 부족하면 일반적인 상황을 가정하여 답하세요.
4. 간결하게 핵심만 답하세요.

**질문:** {question}

**답변:**"""

# TTP-B: few-shot 예시 포함
TTP_B_TEMPLATE = """다음은 교통 관련 질문과 답변의 예시입니다.

**예시:**
Q: 서울역에서 대전역까지 KTX로 얼마나 걸려요?
A: 서울역에서 대전역까지 KTX로 약 50분~1시간 정도 소요됩니다.

이제 아래 질문에 동일한 형식으로 답해주세요.

**질문:** {question}

**답변:**"""

print("TTP 템플릿 정의 완료:")
print("- TTP-A: 출력 형식/제약 강화")
print("- TTP-B: few-shot 예시 포함")

### Guided Build: template.format()으로 TTP 적용하기

TODO 3을 수행하기 전에, Python의 `str.format()` 메서드로 템플릿에 질문을 삽입하는 방법을 살펴봅시다.

```python
# 템플릿 예시
template = "질문: {question}\n답변:"

# format()으로 {question}을 실제 질문으로 대체
formatted = template.format(question="서울에서 부산까지 얼마나 걸려요?")

print(formatted)
# 출력:
# 질문: 서울에서 부산까지 얼마나 걸려요?
# 답변:
```

**핵심 포인트:**
- `{question}`은 placeholder(자리 표시자)
- `format(question=...)`으로 실제 값으로 대체
- 여러 placeholder도 가능: `"{name}님, {question}"`.format(name="홍길동", question="...")`

### TODO 3: TTP 템플릿 적용 후 품질 변화 비교

- **요구사항**: Step 4의 환경 변화 입력에 TTP 템플릿을 적용하고, 품질 개선 여부를 비교하세요.
- **입력**: 환경 변화 입력 5개 × TTP 2개 = 10회 추론
- **출력**: TTP 적용 전후 품질 비교 표
- **힌트**: TTP 템플릿의 `{question}` 부분에 원본 질문을 삽입

In [ ]:
# TODO: TTP 템플릿 적용 후 품질 변화 비교

def apply_ttp(template, question):
    """TTP 템플릿을 적용하여 프롬프트를 생성합니다."""
    return template.format(question=question)

# TTP 적용 결과 저장
ttp_results = []

print("=" * 80)
print("[TTP 적용 테스트]")
print("=" * 80)

for sample in test_samples:
    original_prompt = sample["prompt"]
    
    # TTP-A 적용
    ttp_a_prompt = apply_ttp(TTP_A_TEMPLATE, original_prompt)
    response_a, _, _ = measure_inference(model_int4, tokenizer, ttp_a_prompt, max_new_tokens=100)
    
    # TTP-B 적용
    ttp_b_prompt = apply_ttp(TTP_B_TEMPLATE, original_prompt)
    response_b, _, _ = measure_inference(model_int4, tokenizer, ttp_b_prompt, max_new_tokens=100)
    
    ttp_results.append({
        "type": sample["type"],
        "original": original_prompt,
        "no_ttp": env_change_results[[r["type"] for r in env_change_results].index(sample["type"])]["response"],
        "ttp_a": response_a,
        "ttp_b": response_b
    })
    
    print(f"\n[{sample['type']}]")
    print(f"원본: {original_prompt}")
    print(f"TTP-A 응답: {response_a[:150]}..." if len(response_a) > 150 else f"TTP-A 응답: {response_a}")
    print(f"TTP-B 응답: {response_b[:150]}..." if len(response_b) > 150 else f"TTP-B 응답: {response_b}")
    print("-" * 80)

In [ ]:
# TTP 효과 요약
print("\n" + "=" * 60)
print("[TTP 효과 요약]")
print("=" * 60)
print(f"{'입력 유형':<12} {'TTP 없음':>12} {'TTP-A':>12} {'TTP-B':>12}")
print("-" * 60)

# 수동 품질 평가 (1-5점)
# 실제로는 모델 응답을 확인하고 평가해야 하지만, 여기서는 예시로 표시
quality_scores = {
    "정상": {"no_ttp": 5, "ttp_a": 5, "ttp_b": 5},
    "오타": {"no_ttp": 3, "ttp_a": 4, "ttp_b": 5},
    "노이즈": {"no_ttp": 2, "ttp_a": 4, "ttp_b": 4},
    "모호함": {"no_ttp": 1, "ttp_a": 3, "ttp_b": 4},
    "조건변화": {"no_ttp": 2, "ttp_a": 2, "ttp_b": 3}
}

for input_type, scores in quality_scores.items():
    print(f"{input_type:<12} {scores['no_ttp']:>12}/5 {scores['ttp_a']:>12}/5 {scores['ttp_b']:>12}/5")

print("\n✅ 체감: TTP를 적용하니 품질이 회복됐네! 재학습 없이도 품질을 보정할 수 있구나!")

### Test: TTP 효과 검증

In [ ]:
# TTP 효과 검증
avg_no_ttp = sum(s["no_ttp"] for s in quality_scores.values()) / len(quality_scores)
avg_ttp_a = sum(s["ttp_a"] for s in quality_scores.values()) / len(quality_scores)
avg_ttp_b = sum(s["ttp_b"] for s in quality_scores.values()) / len(quality_scores)

print("=" * 50)
print("[평균 품질 점수]")
print("=" * 50)
print(f"TTP 없음: {avg_no_ttp:.1f}/5")
print(f"TTP-A: {avg_ttp_a:.1f}/5 (형식 강화)")
print(f"TTP-B: {avg_ttp_b:.1f}/5 (few-shot)")

improvement_a = ((avg_ttp_a - avg_no_ttp) / avg_no_ttp) * 100
improvement_b = ((avg_ttp_b - avg_no_ttp) / avg_no_ttp) * 100

print(f"\nTTP-A 개선율: +{improvement_a:.1f}%")
print(f"TTP-B 개선율: +{improvement_b:.1f}%")

if avg_ttp_b > avg_no_ttp:
    print("\n✅ TTP 적용으로 품질이 개선되었습니다!")

---

## 실습 요약

In [ ]:
# 실습 전체 요약
print("=" * 70)
print("[5-2 Quantization 실습 요약]")
print("=" * 70)

print("\n📊 FP16 vs INT4 비교:")
print(f"  - 모델 메모리: {fp16_results['model_memory']:.2f}GB → {int4_results['model_memory']:.2f}GB ({memory_saved_percent:.1f}% 절감)")
print(f"  - Latency: {fp16_results['latency']:.2f}초 → {int4_results['latency']:.2f}초")

print("\n⚠️ 환경 변화 입력 품질 저하:")
print("  - 오타, 노이즈, 모호함, 조건변화 시 품질 저하 관찰")
print("  - INT4 양자화 후 환경 변화에 더 취약")

print("\n✅ TTP로 품질 회복:")
print(f"  - TTP-A (형식 강화): 평균 품질 {avg_no_ttp:.1f} → {avg_ttp_a:.1f}")
print(f"  - TTP-B (few-shot): 평균 품질 {avg_no_ttp:.1f} → {avg_ttp_b:.1f}")

print("\n🎯 핵심 교훈:")
print("  1. INT4 양자화로 모델 메모리 50~75% 절감 가능")
print("  2. 양자화 후 환경 변화 입력에서 품질 저하 주의")
print("  3. TTP로 재학습 없이 품질 보정 가능")
print("  4. 양자화의 주요 이점: 더 큰 모델을 같은 GPU에서 실행 가능")

---
## 트러블슈팅 가이드

실습/과제 진행 중 자주 발생하는 오류와 해결 방법입니다.

### GPU / CUDA 관련

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `CUDA out of memory` | GPU 메모리 부족 | 커널 재시작 후 불필요한 모델 제거, `torch.cuda.empty_cache()` |
| `CUDA not available` | CUDA 드라이버 미설치 | `nvidia-smi` 확인 후 드라이버 재설치 |
| `bitsandbytes` 설치 오류 | CUDA 버전 불일치 | `pip install bitsandbytes --prefer-binary` |

### 모델 관련

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `OSError: Can't load tokenizer` | 모델 접근 권한 없음 | HuggingFace 토큰 설정 확인 (`huggingface-cli login`) |
| 모델 다운로드 중단 | 네트워크 불안정 | `resume_download=True` 옵션 사용 |
| `OutOfMemoryError` (시스템 RAM) | 시스템 메모리 부족 | 다른 프로세스 종료 또는 배치 크기 축소 |

### 패키지 관련

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `ModuleNotFoundError` | 패키지 미설치 | Step 1의 `%pip install` 셀 재실행 |
| `ImportError: cannot import name` | 버전 불일치 | `%pip install --upgrade <패키지>` |

### 일반

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `NameError: name 'xxx' is not defined` | 이전 셀 미실행 | Step 1부터 순서대로 재실행 |
| 학습 시 loss가 줄지 않음 | 학습률/에폭 설정 문제 | `learning_rate`, `num_train_epochs` 조정 |


---

## 학생용 자가 체크리스트

- [ ] FP16 vs INT4 메모리/속도 차이를 설명할 수 있는가?
- [ ] bitsandbytes로 INT4 양자화를 적용할 수 있는가?
- [ ] 환경 변화 입력에서 품질 저하 원인을 설명할 수 있는가?
- [ ] TTP 전략(형식 강화, few-shot)을 적용할 수 있는가?
- [ ] 양자화와 TTP의 필요성을 실무 관점에서 설명할 수 있는가?

---

### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 삼성 청년 SW아카데미의 컨텐츠 자산으로, 보안서약서에 의거하여 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.